# Categorical Encoding

This notebook handles encoding of categorical variables for machine learning models.

In [5]:
# Imports
import pandas as pd
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import joblib

# Load preprocessed data
df = pd.read_csv('../data/preprocessed_data.csv')
print('Preprocessed data loaded, shape:', df.shape)
df.head()

Preprocessed data loaded, shape: (10999, 12)


,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N
0,1,D,Flight,0.0,2,-0.451220,0.0,low,F,2.0,-0.908270,1
1,2,F,Flight,0.0,5,0.024390,-1.0,low,M,2.0,-0.330478,1
2,3,A,Flight,-1.0,2,-0.378049,1.0,low,M,2.0,-0.241395,1
3,4,B,Flight,-0.5,3,-0.463415,1.0,medium,M,0.5,-0.925713,1
4,5,C,Flight,-1.0,2,-0.365854,0.0,medium,F,2.0,-0.518611,1


In [6]:
# Identify categorical columns
categorical_cols = ['Warehouse_block', 'Mode_of_Shipment', 'Product_importance', 'Gender']
print('Categorical columns:', categorical_cols)

# Check unique values
for col in categorical_cols:
    print(f'{col}: {df[col].unique()}')

Categorical columns: ['Warehouse_block', 'Mode_of_Shipment', 'Product_importance', 'Gender']
Warehouse_block: ['D' 'F' 'A' 'B' 'C']
Mode_of_Shipment: ['Flight' 'Ship' 'Road']
Product_importance: ['low' 'medium' 'high']
Gender: ['F' 'M']


In [8]:
# One-hot encoding for nominal variables
nominal_cols = ['Warehouse_block', 'Mode_of_Shipment', 'Gender']
ohe = OneHotEncoder(drop='first', sparse_output=False)
encoded_nominal = ohe.fit_transform(df[nominal_cols])
encoded_nominal_df = pd.DataFrame(encoded_nominal, columns=ohe.get_feature_names_out(nominal_cols))
print('One-hot encoded nominal columns shape:', encoded_nominal_df.shape)
encoded_nominal_df.head()

One-hot encoded nominal columns shape: (10999, 7)


,Warehouse_block_B,Warehouse_block_C,Warehouse_block_D,Warehouse_block_F,Mode_of_Shipment_Road,Mode_of_Shipment_Ship,Gender_M
0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,1.0,0.0,0.0,1.0
2,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [9]:
# Ordinal encoding for Product_importance
importance_mapping = {'low': 0, 'medium': 1, 'high': 2}
df['Product_importance_encoded'] = df['Product_importance'].map(importance_mapping)
print('Ordinal encoding for Product_importance:')
print(df[['Product_importance', 'Product_importance_encoded']].head())

Ordinal encoding for Product_importance:
  Product_importance  Product_importance_encoded
0                low                           0
1                low                           0
2                low                           0
3             medium                           1
4             medium                           1


In [10]:
# Combine encoded features
df_encoded = df.drop(columns=categorical_cols)
df_encoded = pd.concat([df_encoded, encoded_nominal_df], axis=1)
print('Final encoded dataframe shape:', df_encoded.shape)
df_encoded.head()

Final encoded dataframe shape: (10999, 16)


,ID,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Discount_offered,Weight_in_gms,Reached.on.Time_Y.N,Product_importance_encoded,Warehouse_block_B,Warehouse_block_C,Warehouse_block_D,Warehouse_block_F,Mode_of_Shipment_Road,Mode_of_Shipment_Ship,Gender_M
0,1,0.0,2,-0.451220,0.0,2.0,-0.908270,1,0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,2,0.0,5,0.024390,-1.0,2.0,-0.330478,1,0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
2,3,-1.0,2,-0.378049,1.0,2.0,-0.241395,1,0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,-0.5,3,-0.463415,1.0,0.5,-0.925713,1,1,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,-1.0,2,-0.365854,0.0,2.0,-0.518611,1,1,0.0,1.0,0.0,0.0,0.0,0.0,0.0


In [11]:
# Save encoders and encoded data
joblib.dump(ohe, '../models/onehot_encoder.pkl')
df_encoded.to_csv('../data/encoded_data.csv', index=False)
print('Encoded data saved to ../data/encoded_data.csv')
print('One-hot encoder saved to ../models/onehot_encoder.pkl')

Encoded data saved to ../data/encoded_data.csv
One-hot encoder saved to ../models/onehot_encoder.pkl
